# DPO Дообучение + Оценка

**План:**
1. DPO дообучение на более крупном сэмпле поверх SFT чекпоинта (beta=0.1)
2. Сравнение 5 версий: BASE / SFT / DPO (разные b) на ruMMLU и вручную




In [ ]:
!pip install -q trl transformers datasets peft bitsandbytes accelerate -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 160.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 47.7 MB/s eta 0:00:00


In [ ]:
import json
import gc
import os
import shutil
from pathlib import Path
from typing import Optional
from datasets import load_dataset

import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import DPOTrainer, DPOConfig
from tqdm.auto import tqdm

print('Импорты загружены')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет"}')

Импорты загружены
GPU: NVIDIA A100-SXM4-40GB


In [ ]:
from huggingface_hub import login
login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Модели
BASE_MODEL_ID  = "google/gemma-3-4b-it"
SFT_CHECKPOINT = "/content/drive/MyDrive/Colab Notebooks/gemma3_sft/checkpoints/epoch06_hard_ep2of2"

# Пути
DPO_DATA_PATH  = "/content/drive/MyDrive/Colab Notebooks/dpo_2250.json"
DPO_OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/gemma3_dpo"
RESULTS_DIR    = "/content/drive/MyDrive/mera_results_dpo"
os.makedirs(DPO_OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# DPO гиперпараметры
BETA_VALUES       = [0.01, 0.1, 0.5]
BETA_DEFAULT      = 0.1
DPO_EPOCHS        = 2
DPO_LR            = 1e-5
DPO_BATCH         = 1
DPO_GRAD_ACCUM    = 8
DPO_MAX_LENGTH    = 512

# Оценка
MAX_EVAL_SAMPLES  = 1000
NUM_FEWSHOT       = 5
ANSWER_LETTERS    = ["A", "B", "C", "D"]

print('Конфигурация загружена')
print(f'SFT чекпоинт: {SFT_CHECKPOINT}')

Конфигурация загружена
SFT чекпоинт: /content/drive/MyDrive/Colab Notebooks/gemma3_sft/checkpoints/epoch06_hard_ep2of2


## 1. Подготовка DPO датасета

In [ ]:
with open(DPO_DATA_PATH) as f:
    raw_data = json.load(f)

#если не добавить пробел, токенизатор сходит с ума,т.к. в ответе одна буква только
def fix_whitespace(item):
    return {
        "prompt":   item["prompt"],
        "chosen":   " " + item["chosen"].strip(),
        "rejected": " " + item["rejected"].strip(),
    }

raw_data_fixed = [fix_whitespace(item) for item in raw_data]
dpo_dataset = Dataset.from_list(raw_data_fixed)

# Разбивка train/test
dpo_dataset = dpo_dataset.train_test_split(test_size=0.1, seed=42)

dpo_train = dpo_dataset["train"]
dpo_test  = dpo_dataset["test"]

print(f"Train: {len(dpo_train)}, Test: {len(dpo_test)}")
print(f"\nПример:")
print(dpo_train[0])

Train: 2025, Test: 225

Пример:
{'prompt': 'Lex iniusta non est lex" имеет какое из следующих значений?\nA) Закон не имеет силы до тех пор, пока он официально не принят.\nB) Закон имеет лексический приоритет над моралью.\nC) Несправедливый закон - это не закон.\nD) Никто не стоит выше закона.', 'chosen': ' C', 'rejected': ' A'}


## 2. Загрузка SFT модели

In [ ]:
def load_model_for_dpo(checkpoint_path: str):
    """Загружает модель в 4bit для DPO обучения"""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)

    model = AutoModelForCausalLM.from_pretrained(
        checkpoint_path,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model.config.use_cache = False

    return model, tokenizer


def add_lora_for_dpo(model):
    """Добавляет LoRA адаптер для DPO"""
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    return get_peft_model(model, lora_config)

## 3. DPO обучение (основной запуск beta=0.1)

In [ ]:
# Функция DPO обучения
def run_dpo(
    sft_checkpoint: str,
    train_dataset: Dataset,
    beta: float = 0.1,
    num_epochs: int = 1,
    output_suffix: str = "",
) -> str:
    """
    Запускает DPO обучение поверх SFT чекпоинта.
    Возвращает путь к сохранённому чекпоинту.
    """
    print(f'DPO обучение: beta={beta}, epochs={num_epochs}')
    print()

    output_dir = f"{DPO_OUTPUT_DIR}/beta_{str(beta).replace('.', '_')}{output_suffix}"
    tmp_dir    = f"/content/tmp_dpo_beta_{str(beta).replace('.', '_')}"

    model, tokenizer = load_model_for_dpo(sft_checkpoint)
    model = add_lora_for_dpo(model)

    dpo_config = DPOConfig(
        output_dir                   = tmp_dir,
        num_train_epochs             = num_epochs,
        per_device_train_batch_size  = DPO_BATCH,
        gradient_accumulation_steps  = DPO_GRAD_ACCUM,
        learning_rate                = DPO_LR,
        beta                         = beta,
        bf16                         = True,
        gradient_checkpointing       = True,
        gradient_checkpointing_kwargs= {"use_reentrant": False},
        logging_steps                = 5,
        save_strategy                = "no",
        eval_strategy                = "no",
        report_to                    = "none",
        optim                        = "adamw_torch_fused",
        remove_unused_columns        = False,
        seed                         = 42,
    )

    trainer = DPOTrainer(
        model            = model,
        args             = dpo_config,
        train_dataset    = train_dataset,
        processing_class = tokenizer,
    )

    trainer.train()

    # Сохраняем
    print(f'Сохраняем чекпоинт: {output_dir}')
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    # Чистим память
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

    if Path(tmp_dir).exists():
        shutil.rmtree(tmp_dir, ignore_errors=True)

    print(f'DPO чекпоинт сохранён: {output_dir}')
    return output_dir

In [ ]:
#Пробный запуск DPO (beta=0.1, 3 эпохи)
dpo_main_path = run_dpo(
    sft_checkpoint = SFT_CHECKPOINT,
    train_dataset  = dpo_train,
    beta           = 0.1,
    num_epochs     = DPO_EPOCHS,
)
print(f'Основной DPO чекпоинт: {dpo_main_path}')

DPO обучение: beta=0.1, epochs=3



Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Adding EOS to train dataset:   0%|          | 0/2025 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2025 [00:00<?, ? examples/s]

Step,Training Loss
5,0.693445
10,0.691297
15,0.687743
20,0.683208
25,0.679005
30,0.670834
35,0.675064
40,0.662463
45,0.667569
50,0.655082


KeyboardInterrupt: 

Тут у нас кончились ресурсы, и дообучение пришлось прервать, но заодно мы поняли по лоссу, что, кажется, нет смыла дообучать 3 эпохи, 2-ух хватит.

In [ ]:
#Основной запуск DPO (beta=0.1, 2 эпохи)
dpo_main_path = run_dpo(
    sft_checkpoint = SFT_CHECKPOINT,
    train_dataset  = dpo_train,
    beta           = 0.1,
    num_epochs     = DPO_EPOCHS,
)
print(f'Основной DPO чекпоинт: {dpo_main_path}')

DPO обучение: beta=0.1, epochs=2



Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Adding EOS to train dataset:   0%|          | 0/2025 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2025 [00:00<?, ? examples/s]

Step,Training Loss
5,0.693367
10,0.690514
15,0.685935
20,0.683883
25,0.676551
30,0.668434
35,0.672756
40,0.659338
45,0.665911
50,0.650957


Сохраняем чекпоинт: /content/drive/MyDrive/Colab Notebooks/gemma3_dpo/beta_0_1
DPO чекпоинт сохранён: /content/drive/MyDrive/Colab Notebooks/gemma3_dpo/beta_0_1
Основной DPO чекпоинт: /content/drive/MyDrive/Colab Notebooks/gemma3_dpo/beta_0_1


## 5. Оценка на ruMMLU

In [ ]:
mera = load_dataset("ai-forever/MERA", name="rummlu")
public_df = mera["public_test"].to_pandas()

# domain спрятан в meta, достаём его, чтобы отобрать только естественные науки
public_df["domain"] = public_df["meta"].apply(
    lambda m: m.get("domain") if isinstance(m, dict) else None
)


SCIENCE_DOMAINS = [
    "anatomy", "astronomy", "college_biology", "college_chemistry",
    "college_medicine", "college_physics", "conceptual_physics",
    "high_school_biology", "high_school_chemistry", "high_school_physics",
    "medical_genetics", "nutrition", "professional_medicine", "virology",
    "high_school_geography",
]

def filter_natural_sciences(df):
    return df[df["domain"].isin(SCIENCE_DOMAINS)].reset_index(drop=True)

print(f'Всего примеров в ruMMLU: {len(public_df)}')
print(f'Естественных наук: {len(filter_natural_sciences(public_df))}')

README.md:   0%|          | 0.00/137k [00:00<?, ?B/s]

data/rummlu/train.jsonl:   0%|          | 0.00/13.3M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/845k [00:00<?, ?B/s]

Generating public_test split:   0%|          | 0/10033 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/961 [00:00<?, ? examples/s]

Всего примеров в ruMMLU: 10033
Естественных наук: 2033


In [ ]:
with open(DPO_DATA_PATH) as f:
    raw_data = json.load(f)

dpo_prompts = set(item["prompt"] for item in raw_data)
print(f"DPO промптов: {len(dpo_prompts)}")

# Фильтруем ruMMLU, т.е. убираем те что были в обучении
def format_prompt_for_matching(row):
    inputs = row["inputs"] if isinstance(row["inputs"], dict) else {}
    return (
        f"{inputs.get('text', '')}\n"
        f"A) {inputs.get('option_a', '')}\n"
        f"B) {inputs.get('option_b', '')}\n"
        f"C) {inputs.get('option_c', '')}\n"
        f"D) {inputs.get('option_d', '')}"
    )

public_df["_prompt"] = public_df.apply(format_prompt_for_matching, axis=1)
clean_eval_df = public_df[~public_df["_prompt"].isin(dpo_prompts)].drop(columns=["_prompt"])

print(f"Всего в ruMMLU:         {len(public_df)}")
print(f"Было в DPO обучении:    {len(public_df) - len(clean_eval_df)}")
print(f"Чистых для оценки:      {len(clean_eval_df)}")

DPO промптов: 2096
Всего в ruMMLU:         10033
Было в DPO обучении:    2096
Чистых для оценки:      7937


In [ ]:
#Функции оценки
def build_eval_prompt(row: pd.Series, fewshot_pool: pd.DataFrame) -> str:
    """Строит few-shot промпт для оценки"""
    def format_example(r, with_answer=True):
        instruction = r["instruction"]
        prompt = instruction.format(
            subject=r.get("subject", ""),
            text=r.get("text", ""),
            option_a=r.get("option_a", ""),
            option_b=r.get("option_b", ""),
            option_c=r.get("option_c", ""),
            option_d=r.get("option_d", ""),
        )
        if with_answer:
            prompt += f" {str(r.get('outputs', '')).strip()}"
        return prompt

    domain     = row.get("domain", "")
    current_id = row["meta"].get("id") if isinstance(row.get("meta"), dict) else None
    pool = fewshot_pool[fewshot_pool["domain"] == domain]
    if current_id is not None:
        pool = pool[pool["meta"].apply(
            lambda m: m.get("id") if isinstance(m, dict) else None
        ) != current_id]

    shots = pool.sample(min(NUM_FEWSHOT, len(pool)), random_state=42)
    prompt = ""
    for _, shot in shots.iterrows():
        prompt += format_example(shot, with_answer=True) + "\n\n"
    prompt += format_example(row, with_answer=False)
    return prompt





#Оценка через логиты
def evaluate_with_logits(
    model,
    tokenizer,
    eval_df: pd.DataFrame,
    fewshot_pool: pd.DataFrame,
    label: str = "model",
) -> pd.DataFrame:
    """
    Оценка через логиты честнее для модели обученной на развёрнутых ответах.
    Смотрим на вероятности токенов A/B/C/D напрямую.
    """
    # Токены для A B C D (латиница + кириллица)
    token_ids = {}
    for letter, cyrillic in [("A","А"), ("B","В"), ("C","С"), ("D","Д")]:
        ids_lat = tokenizer.encode(letter, add_special_tokens=False)
        ids_cyr = tokenizer.encode(cyrillic, add_special_tokens=False)
        token_ids[letter] = ids_lat + ids_cyr

    results = []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=f"Eval [{label}]"):
        prompt = build_eval_prompt(row, fewshot_pool)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)

        with torch.no_grad():
            logits = model(**inputs).logits[0, -1, :]

        scores    = {letter: max(logits[tid].item() for tid in tids)
                     for letter, tids in token_ids.items()}
        predicted = max(scores, key=scores.get)
        gold      = str(row.get("outputs", "")).strip().upper()
        meta_id   = row["meta"].get("id") if isinstance(row.get("meta"), dict) else None

        results.append({
            "id":        meta_id,
            "domain":    row.get("domain", ""),
            "subject":   row.get("subject", ""),
            "gold":      gold,
            "predicted": predicted,
            "correct":   predicted == gold,
            "scores":    str(scores),
        })

    return pd.DataFrame(results)

In [ ]:
def print_results(df: pd.DataFrame, label: str = ""):
    acc = df["correct"].mean()
    print(f"\n{'='*55}")
    print(f"Результаты: {label}")
    print(f"{'='*55}")
    print(f"Accuracy: {acc:.4f} ({acc*100:.1f}%)  |  n={len(df)}")
    by_domain = (
        df.groupby("domain")["correct"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "accuracy", "count": "n"})
        .sort_values("accuracy", ascending=False)
    )
    print(by_domain.to_string(float_format=lambda x: f"{x:.3f}"))
    return acc





def load_model_for_eval(model_path: str, label: str = ""):
    """Загружает модель для оценки (4bit)"""
    print(f"\nЗагружаем [{label}]: {model_path}")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model.eval()
    return model, tokenizer





def generate_answer(model, tokenizer, prompt: str, max_new_tokens: int = 5) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
model_dpo, tok_dpo = load_model_for_eval(dpo_main_path, label="DPO beta=0.1")

eval_df = clean_eval_df.sample(
    min(MAX_EVAL_SAMPLES, len(clean_eval_df)), random_state=42
).reset_index(drop=True)
eval_sci = filter_natural_sciences(eval_df)

print(f"Всего для оценки:  {len(eval_df)}")
print(f"Естественных наук: {len(eval_sci)}")

for _, row in eval_df.head(20).iterrows():
    prompt = build_eval_prompt(row, clean_eval_df)
    answer = generate_answer(model_dpo, tok_dpo, prompt, max_new_tokens=5)
    print(f"gold={row['outputs']}  generated={repr(answer.strip())}")


Загружаем [DPO beta=0.1]: /content/drive/MyDrive/Colab Notebooks/gemma3_dpo/beta_0_1


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Всего для оценки:  1000
Естественных наук: 166


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


gold=A  generated='C'
gold=C  generated='C'
gold=C  generated='D'
gold=C  generated='C'
gold=A  generated='A'
gold=A  generated='C'
gold=B  generated='C'
gold=A  generated='C'
gold=A  generated='B'
gold=C  generated='А'
gold=A  generated='A'
gold=B  generated='C'
gold=D  generated='A'
gold=A  generated='C'
gold=D  generated='C'
gold=A  generated='A'
gold=B  generated='C'
gold=B  generated='B'
gold=D  generated='C'
gold=C  generated='C'


Уже генерирует разные буквы!!!
Но не те((((
  

In [ ]:
# Генерация
gen_correct = 0
logit_correct = 0

token_ids = {}
for letter, cyrillic in [("A","А"), ("B","В"), ("C","С"), ("D","Д")]:
    ids_lat = tok_dpo.encode(letter, add_special_tokens=False)
    ids_cyr = tok_dpo.encode(cyrillic, add_special_tokens=False)
    token_ids[letter] = ids_lat + ids_cyr

for _, row in eval_df.head(100).iterrows():
    prompt = build_eval_prompt(row, clean_eval_df)
    inputs = tok_dpo(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model_dpo.device)
    gold = str(row["outputs"]).strip().upper()

    # Генерация
    with torch.no_grad():
        out = model_dpo.generate(**inputs, max_new_tokens=3, do_sample=False,
                                  pad_token_id=tok_dpo.eos_token_id)
    gen_pred = tok_dpo.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()[0].upper()
    gen_correct += (gen_pred == gold)

    # Логиты
    with torch.no_grad():
        logits = model_dpo(**inputs).logits[0, -1, :]
    scores = {l: max(logits[tid].item() for tid in tids) for l, tids in token_ids.items()}
    logit_pred = max(scores, key=scores.get)
    logit_correct += (logit_pred == gold)

print(f"Генерация: {gen_correct/100:.1%}")
print(f"Логиты:    {logit_correct/100:.1%}")

Генерация: 24.0%
Логиты:    25.0%


По логитам как будто лучше, поэтому пробуем все через логиты оценивать.

Спойлер: потом мы посмотрим на результаты, и решим, что на генерации тоже надо оценить.


In [ ]:
model_base, tok_base = load_model_for_eval(BASE_MODEL_ID, label="BASE")

base_correct = 0
for _, row in eval_df.head(100).iterrows():
    prompt = build_eval_prompt(row, clean_eval_df)
    inputs = tok_base(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model_base.device)
    gold = str(row["outputs"]).strip().upper()
    with torch.no_grad():
        logits = model_base(**inputs).logits[0, -1, :]
    scores = {l: max(logits[tid].item() for tid in tids) for l, tids in token_ids.items()}
    pred = max(scores, key=scores.get)
    base_correct += (pred == gold)

print(f"BASE на тех же 100: {base_correct/100:.1%}")
del model_base, tok_base
gc.collect(); torch.cuda.empty_cache()


Загружаем [BASE]: google/gemma-3-4b-it


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

BASE на тех же 100: 23.0%


In [ ]:
import json
from collections import Counter

with open(DPO_DATA_PATH) as f:
    raw_data = json.load(f)

print("chosen:", Counter(d["chosen"] for d in raw_data))
print("rejected:", Counter(d["rejected"][0] for d in raw_data))

chosen: Counter({'D': 618, 'C': 554, 'A': 551, 'B': 527})
rejected: Counter({'A': 563, 'B': 563, 'D': 562, 'C': 562})


In [ ]:
MAX_EVAL_SAMPLES = 1000

In [ ]:
#Выборка для оценки
eval_df = clean_eval_df.sample(
    min(MAX_EVAL_SAMPLES, len(clean_eval_df)), random_state=42
).reset_index(drop=True)
eval_sci = filter_natural_sciences(eval_df)

print(f'Всего для оценки:      {len(eval_df)}')
print(f'Естественных наук:     {len(eval_sci)}')

Всего для оценки:      1000
Естественных наук:     166


In [ ]:
# Сравнение двух вариантов промпта (с фью-шотом и без)

def build_prompt_v2(row: pd.Series, fewshot_pool: pd.DataFrame) -> str:
    return (
        f"Выберите правильный ответ (ответьте только буквой A, B, C или D):\n\n"
        f"{row.get('text', '')}\n\n"
        f"A) {row.get('option_a', '')}\n"
        f"B) {row.get('option_b', '')}\n"
        f"C) {row.get('option_c', '')}\n"
        f"D) {row.get('option_d', '')}\n\n"
        f"Ответ:"
    )


def evaluate_with_logits_prompt(
    model,
    tokenizer,
    eval_df: pd.DataFrame,
    fewshot_pool: pd.DataFrame,
    prompt_fn,
    label: str = "model",
) -> pd.DataFrame:
    token_ids = {}
    for letter, cyrillic in [("A","А"), ("B","В"), ("C","С"), ("D","Д")]:
        ids_lat = tokenizer.encode(letter, add_special_tokens=False)
        ids_cyr = tokenizer.encode(cyrillic, add_special_tokens=False)
        token_ids[letter] = ids_lat + ids_cyr

    results = []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=f"Eval [{label}]"):
        prompt = prompt_fn(row, fewshot_pool)
        inputs = tokenizer(
            prompt, return_tensors="pt", truncation=True, max_length=2048
        ).to(model.device)

        with torch.no_grad():
            logits = model(**inputs).logits[0, -1, :]

        scores    = {l: max(logits[tid].item() for tid in tids)
                     for l, tids in token_ids.items()}
        predicted = max(scores, key=scores.get)
        gold      = str(row.get("outputs", "")).strip().upper()
        meta_id   = row["meta"].get("id") if isinstance(row.get("meta"), dict) else None

        results.append({
            "id":        meta_id,
            "domain":    row.get("domain", ""),
            "subject":   row.get("subject", ""),
            "gold":      gold,
            "predicted": predicted,
            "correct":   predicted == gold,
        })

    return pd.DataFrame(results)


def compare_prompts(model, tokenizer, eval_df, fewshot_pool, model_label="MODEL"):
    print(f"\n{'='*60}")
    print(f"Сравнение промптов: {model_label}")
    print(f"Примеров: {len(eval_df)}")
    print(f"{'='*60}\n")

    print("Промпт 1: few-shot")
    df_v1 = evaluate_with_logits_prompt(
        model, tokenizer, eval_df, fewshot_pool,
        prompt_fn=build_eval_prompt,
        label=f"{model_label}_v1_fewshot",
    )

    print("Промпт 2: инструкция без few-shot")
    df_v2 = evaluate_with_logits_prompt(
        model, tokenizer, eval_df, fewshot_pool,
        prompt_fn=build_prompt_v2,
        label=f"{model_label}_v2_direct",
    )

    acc_v1 = df_v1["correct"].mean()
    acc_v2 = df_v2["correct"].mean()
    delta  = acc_v2 - acc_v1

    print()
    print(f"{'Промпт':<35} {'Accuracy':>10}")
    print()
    print(f"{'v1: few-shot':<35} {acc_v1*100:>9.1f}%")
    print(f"{'v2: инструкция без few-shot':<35} {acc_v2*100:>9.1f}%")
    print()
    winner = "v2 лучше" if delta > 0 else "v1 лучше" if delta < 0 else "одинаково"
    print(f"{'Разница (v2 - v1)':<35} {delta*100:>+9.1f}%  {winner}")
    print()

    df_v1["prompt"] = "v1_fewshot"
    df_v2["prompt"] = "v2_direct"
    combined = pd.concat([df_v1, df_v2])
    domain_table = (
        combined.groupby(["domain", "prompt"])["correct"]
        .mean()
        .unstack("prompt")
        .rename(columns={"v1_fewshot": "v1_few%", "v2_direct": "v2_direct%"})
        .assign(delta=lambda d: d["v2_direct%"] - d["v1_few%"])
        .sort_values("delta", ascending=False)
        .mul(100)
        .round(1)
    )
    print("Разбивка по доменам:")
    print(domain_table.to_string())
    print()

    return df_v1, df_v2


# ЗАПУСК
model_base, tok_base = load_model_for_eval(BASE_MODEL_ID, label="BASE")

base_v1, base_v2 = compare_prompts(
    model_base, tok_base,
    eval_df=eval_df,
    fewshot_pool=clean_eval_df,
    model_label="BASE",
)

del model_base, tok_base
gc.collect(); torch.cuda.empty_cache()



model_sft, tok_sft = load_model_for_eval(SFT_CHECKPOINT, label="SFT")

sft_v1, sft_v2 = compare_prompts(
    model_sft, tok_sft,
    eval_df=eval_df,
    fewshot_pool=clean_eval_df,
    model_label="SFT",
)

del model_sft, tok_sft
gc.collect(); torch.cuda.empty_cache()


DPO_CHECKPOINT = "/content/drive/MyDrive/Colab Notebooks/gemma3_dpo/beta_0_1"
model_dpo, tok_dpo = load_model_for_eval(DPO_CHECKPOINT, label="DPO beta=0.1")

dpo_v1, dpo_v2 = compare_prompts(
    model_dpo, tok_dpo,
    eval_df=eval_df,
    fewshot_pool=clean_eval_df,
    model_label="DPO beta=0.1",
)

del model_dpo, tok_dpo
gc.collect(); torch.cuda.empty_cache()



print()
print("ИТОГ: какой промпт лучше для каждой модели")
print()
for label, v1, v2 in [("BASE", base_v1, base_v2), ("SFT", sft_v1, sft_v2), ("DPO beta=0.1", dpo_v1, dpo_v2)]:
    a1, a2 = v1["correct"].mean()*100, v2["correct"].mean()*100
    print(f"{label:<15}  v1(few-shot): {a1:.1f}%   v2(direct): {a2:.1f}%   Δ={a2-a1:+.1f}%")


Загружаем [BASE]: google/gemma-3-4b-it


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Сравнение промптов: BASE
Примеров: 1000

Промпт 1: few-shot


Eval [BASE_v1_fewshot]:   0%|          | 0/1000 [00:00<?, ?it/s]

...


Сравнение промптов: DPO beta=0.1
Примеров: 1000

Промпт 1: few-shot


Eval [DPO beta=0.1_v1_fewshot]:   0%|          | 0/1000 [00:00<?, ?it/s]

Промпт 2: инструкция без few-shot


Eval [DPO beta=0.1_v2_direct]:   0%|          | 0/1000 [00:00<?, ?it/s]


Промпт                                Accuracy

v1: few-shot                             23.7%
v2: инструкция без few-shot              24.9%

Разница (v2 - v1)                        +1.2%  v2 лучше

Разбивка по доменам:
prompt                               v1_few%  v2_direct%  delta
domain                                                         
college_chemistry                        0.0       100.0  100.0
machine_learning                        20.0        60.0   40.0
medical_genetics                        33.3        66.7   33.3
high_school_government_and_politics      7.1        35.7   28.6
professional_psychology                 11.1        35.6   24.4
college_biology                         11.1        33.3   22.2
formal_logic                            10.0        30.0   20.0
moral_disputes                          13.0        32.6   19.6
high_school_geography                   11.8        29.4   17.6
us_foreign_policy                       33.3        50.0   16.7
college_m

Может показаться, что лучше без фью-шота, но тогда там одинаковое качество у всех моделей. Это потому что они просто везде предсказывают А))) Мы это проверили.

Надо брать с фью-шотом



In [ ]:
#Оценка BASE модели
model_base, tok_base = load_model_for_eval(BASE_MODEL_ID, label="BASE")

base_all_df = evaluate_with_logits(model_base, tok_base, eval_df, clean_eval_df, label="BASE_all")
base_sci_df = evaluate_with_logits(model_base, tok_base, eval_sci, clean_eval_df, label="BASE_sci")

base_acc_all = print_results(base_all_df, "BASE все предметы")
base_acc_sci = print_results(base_sci_df, "BASE естественные науки")

base_all_df.to_csv(f"{RESULTS_DIR}/base_all.csv", index=False)
base_sci_df.to_csv(f"{RESULTS_DIR}/base_sci.csv", index=False)

del model_base, tok_base
gc.collect(); torch.cuda.empty_cache()


Загружаем [BASE]: google/gemma-3-4b-it


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Eval [BASE_all]:   0%|          | 0/1000 [00:00<?, ?it/s]

Eval [BASE_sci]:   0%|          | 0/166 [00:00<?, ?it/s]


Результаты: BASE все предметы
Accuracy: 0.2330 (23.3%)  |  n=1000
                                     accuracy   n
domain                                           
college_computer_science                1.000   1
high_school_european_history            0.500   8
college_physics                         0.500   4
human_sexuality                         0.500   6
philosophy                              0.429  21
high_school_us_history                  0.417  12
machine_learning                        0.400   5
abstract_algebra                        0.400   5
global_facts                            0.400  10
conceptual_physics                      0.385  13
world_religions                         0.385  13
econometrics                            0.375   8
professional_accounting                 0.353  17
high_school_world_history               0.333  15
international_law                       0.333  12
medical_genetics                        0.333   3
security_studies                 

In [ ]:
# Оценка SFT модели
model_sft, tok_sft = load_model_for_eval(SFT_CHECKPOINT, label="SFT")

sft_all_df = evaluate_with_logits(model_sft, tok_sft, eval_df, clean_eval_df, label="SFT_all")
sft_sci_df = evaluate_with_logits(model_sft, tok_sft, eval_sci, clean_eval_df, label="SFT_sci")

sft_acc_all = print_results(sft_all_df, "SFT все предметы")
sft_acc_sci = print_results(sft_sci_df, "SFT естественные науки")

sft_all_df.to_csv(f"{RESULTS_DIR}/sft_all.csv", index=False)
sft_sci_df.to_csv(f"{RESULTS_DIR}/sft_sci.csv", index=False)

del model_sft, tok_sft
gc.collect(); torch.cuda.empty_cache()


Загружаем [SFT]: /content/drive/MyDrive/Colab Notebooks/gemma3_sft/checkpoints/epoch06_hard_ep2of2


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Eval [SFT_all]:   0%|          | 0/1000 [00:00<?, ?it/s]

Eval [SFT_sci]:   0%|          | 0/166 [00:00<?, ?it/s]


Результаты: SFT все предметы
Accuracy: 0.2440 (24.4%)  |  n=1000
                                     accuracy   n
domain                                           
college_computer_science                1.000   1
high_school_european_history            0.625   8
astronomy                               0.500  14
college_physics                         0.500   4
computer_security                       0.500   4
international_law                       0.417  12
high_school_world_history               0.400  15
high_school_biology                     0.391  23
philosophy                              0.381  21
econometrics                            0.375   8
business_ethics                         0.375   8
professional_accounting                 0.353  17
human_sexuality                         0.333   6
elementary_mathematics                  0.333  36
college_medicine                        0.333   3
anatomy                                 0.333   9
conceptual_physics                

In [ ]:
beta_checkpoints = {
    0.01: f"{DPO_OUTPUT_DIR}/beta_0_01",
    0.1:  f"{DPO_OUTPUT_DIR}/beta_0_1",
    0.5:  f"{DPO_OUTPUT_DIR}/beta_0_5",
}

In [ ]:
# Оценка DPO модели
dpo_results = {}

for beta, ckpt_path in sorted(beta_checkpoints.items()):
    model_dpo, tok_dpo = load_model_for_eval(ckpt_path, label=f"DPO beta={beta}")

    dpo_all_df = evaluate_with_logits(model_dpo, tok_dpo, eval_df, clean_eval_df, label=f"DPO_b{beta}_all")
    dpo_sci_df = evaluate_with_logits(model_dpo, tok_dpo, eval_sci, clean_eval_df, label=f"DPO_b{beta}_sci")

    acc_all = print_results(dpo_all_df, f"DPO beta={beta} все предметы")
    acc_sci = print_results(dpo_sci_df, f"DPO beta={beta} естественные науки")

    dpo_all_df.to_csv(f"{RESULTS_DIR}/dpo_beta{str(beta).replace('.','_')}_all.csv", index=False)
    dpo_sci_df.to_csv(f"{RESULTS_DIR}/dpo_beta{str(beta).replace('.','_')}_sci.csv", index=False)

    dpo_results[beta] = {
        "acc_all": acc_all, "acc_sci": acc_sci,
        "all_df": dpo_all_df, "sci_df": dpo_sci_df,
    }

    del model_dpo, tok_dpo
    gc.collect(); torch.cuda.empty_cache()


Загружаем [DPO beta=0.01]: /content/drive/MyDrive/Colab Notebooks/gemma3_dpo/beta_0_01


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Eval [DPO_b0.01_all]:   0%|          | 0/1000 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Eval [DPO_b0.5_all]:   0%|          | 0/1000 [00:00<?, ?it/s]

Eval [DPO_b0.5_sci]:   0%|          | 0/166 [00:00<?, ?it/s]


Результаты: DPO beta=0.5 все предметы
Accuracy: 0.2400 (24.0%)  |  n=1000
                                     accuracy   n
domain                                           
college_computer_science                1.000   1
high_school_european_history            0.625   8
computer_security                       0.500   4
college_physics                         0.500   4
econometrics                            0.500   8
international_law                       0.500  12
high_school_world_history               0.467  15
philosophy                              0.381  21
business_ethics                         0.375   8
astronomy                               0.357  14
professional_accounting                 0.353  17
high_school_biology                     0.348  23
human_sexuality                         0.333   6
college_medicine                        0.333   3
medical_genetics                        0.333   3
us_foreign_policy                       0.333  12
anatomy                  

## 6. Сравнение всех версий

In [ ]:
print("ИТОГОВОЕ СРАВНЕНИЕ") #на 500
print()
print(f"{'Модель':<25} {'Все предметы':>15} {'Естественные науки':>20}")
print("-"*65)
print(f"{'BASE':<25} {base_acc_all*100:>14.1f}% {base_acc_sci*100:>19.1f}%")
print(f"{'SFT':<25} {sft_acc_all*100:>14.1f}% {sft_acc_sci*100:>19.1f}%")
for beta, res in sorted(dpo_results.items()):
    label = f"DPO beta={beta}"
    print(f"{label:<25} {res['acc_all']*100:>14.1f}% {res['acc_sci']*100:>19.1f}%")
print("="*65)

ИТОГОВОЕ СРАВНЕНИЕ

Модель                       Все предметы   Естественные науки
-----------------------------------------------------------------
BASE                                20.4%                21.8%
SFT                                 23.6%                27.6%
DPO beta=0.1                        22.8%                29.9%


In [ ]:
print("ИТОГОВОЕ СРАВНЕНИЕ") # на 1000
print()
print(f"{'Модель':<25} {'Все предметы':>15} {'Естественные науки':>20}")
print("-"*65)
print(f"{'BASE':<25} {base_acc_all*100:>14.1f}% {base_acc_sci*100:>19.1f}%")
print(f"{'SFT':<25} {sft_acc_all*100:>14.1f}% {sft_acc_sci*100:>19.1f}%")
for beta, res in sorted(dpo_results.items()):
    label = f"DPO beta={beta}"
    print(f"{label:<25} {res['acc_all']*100:>14.1f}% {res['acc_sci']*100:>19.1f}%")
print("="*65)

ИТОГОВОЕ СРАВНЕНИЕ

Модель                       Все предметы   Естественные науки
-----------------------------------------------------------------
BASE                                23.3%                20.5%
SFT                                 24.4%                27.1%
DPO beta=0.1                        23.7%                27.1%


In [ ]:
print("ИТОГОВОЕ СРАВНЕНИЕ") # на 1000 все версии
print()
print(f"{'Модель':<25} {'Все предметы':>15} {'Естественные науки':>20}")
print("-"*65)
print(f"{'BASE':<25} {base_acc_all*100:>14.1f}% {base_acc_sci*100:>19.1f}%")
print(f"{'SFT':<25} {sft_acc_all*100:>14.1f}% {sft_acc_sci*100:>19.1f}%")
for beta, res in sorted(dpo_results.items()):
    label = f"DPO beta={beta}"
    print(f"{label:<25} {res['acc_all']*100:>14.1f}% {res['acc_sci']*100:>19.1f}%")
print("="*65)

ИТОГОВОЕ СРАВНЕНИЕ

Модель                       Все предметы   Естественные науки
-----------------------------------------------------------------
BASE                                23.3%                20.5%
SFT                                 24.4%                27.1%
DPO beta=0.01                       24.2%                26.5%
DPO beta=0.1                        23.7%                27.1%
DPO beta=0.5                        24.0%                26.5%


## 7. Ручная оценка (качественный анализ)

5 моделей (BASE, SFT, DPO beta=0.01/0.1/0.5)

In [ ]:

import pandas as pd
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset

BASE_MODEL_ID  = "google/gemma-3-4b-it"
SFT_CHECKPOINT = "/content/drive/MyDrive/Colab Notebooks/gemma3_sft/checkpoints/epoch06_hard_ep2of2"
DPO_CHECKPOINTS = {
    0.01: "/content/drive/MyDrive/Colab Notebooks/gemma3_dpo/beta_0_01",
    0.1:  "/content/drive/MyDrive/Colab Notebooks/gemma3_dpo/beta_0_1",
    0.5:  "/content/drive/MyDrive/Colab Notebooks/gemma3_dpo/beta_0_5",
}
RESULTS_DIR = "/content/drive/MyDrive/mera_results_dpo"

# Параметры просмотра
N_SAMPLES     = 10
DOMAIN_FILTER = None   # например "anatomy" или None = все домены
ONLY_ERRORS   = True   # True = только примеры где хоть одна модель ошиблась
SHOW_SCIENCE  = True   # True = eval_sci, False = eval_all

suffix = "sci" if SHOW_SCIENCE else "all"

base_df    = pd.read_csv(f"{RESULTS_DIR}/base_{suffix}.csv")
sft_df     = pd.read_csv(f"{RESULTS_DIR}/sft_{suffix}.csv")
dpo_dfs    = {
    b: pd.read_csv(f"{RESULTS_DIR}/dpo_beta{str(b).replace('.','_')}_{suffix}.csv")
    for b in [0.01, 0.1, 0.5]
}

print(f"BASE:        {len(base_df)} строк")
print(f"SFT:         {len(sft_df)} строк")
for b, df in dpo_dfs.items():
    print(f"DPO b={b}: {len(df)} строк")

# Строим объединённую таблицу предсказаний
merged = (
    base_df[["id", "domain", "subject", "gold", "predicted"]]
    .rename(columns={"predicted": "base_pred"})
    .merge(sft_df[["id", "predicted"]].rename(columns={"predicted": "sft_pred"}), on="id")
    .merge(dpo_dfs[0.01][["id", "predicted"]].rename(columns={"predicted": "dpo001_pred"}), on="id")
    .merge(dpo_dfs[0.1][["id", "predicted"]].rename(columns={"predicted": "dpo01_pred"}),  on="id")
    .merge(dpo_dfs[0.5][["id", "predicted"]].rename(columns={"predicted": "dpo05_pred"}),  on="id")
)

pred_cols = ["base_pred", "sft_pred", "dpo001_pred", "dpo01_pred", "dpo05_pred"]
for col in pred_cols:
    merged[col + "_ok"] = merged[col] == merged["gold"]

# Фильтры
if DOMAIN_FILTER:
    merged = merged[merged["domain"] == DOMAIN_FILTER].reset_index(drop=True)
    print(f"\nФильтр домена: {DOMAIN_FILTER} ({len(merged)} примеров)")

if ONLY_ERRORS:
    all_correct = merged[[c + "_ok" for c in pred_cols]].all(axis=1)
    merged = merged[~all_correct].reset_index(drop=True)
    print(f"Примеров с хотя бы одной ошибкой: {len(merged)}")

sample_ids = merged["id"].sample(min(N_SAMPLES, len(merged)), random_state=42).tolist()
print(f"Отобрано для показа: {len(sample_ids)}")

print("\nЗагружаем ruMMLU для текста вопросов...")
mera = load_dataset("ai-forever/MERA", name="rummlu")
pub  = mera["public_test"].to_pandas()
pub["id"]     = pub["meta"].apply(lambda m: m.get("id")     if isinstance(m, dict) else None)
pub["domain"] = pub["meta"].apply(lambda m: m.get("domain") if isinstance(m, dict) else None)
inputs_exp    = pd.json_normalize(pub["inputs"].tolist())
pub = pd.concat([pub.drop(columns=["inputs"]), inputs_exp], axis=1)
id_to_row = pub.set_index("id")
print(f"Датасет загружен: {len(pub)} примеров")

NUM_FEWSHOT = 5

def build_eval_prompt_manual(row, fewshot_pool):
    def fmt(r, with_ans=True):
        p = r["instruction"].format(
            subject=r.get("subject", ""),
            text=r.get("text", ""),
            option_a=r.get("option_a", ""),
            option_b=r.get("option_b", ""),
            option_c=r.get("option_c", ""),
            option_d=r.get("option_d", ""),
        )
        if with_ans:
            p += f" {str(r.get('outputs', '')).strip()}"
        return p

    domain  = row.get("domain", "")
    cur_id  = row["meta"].get("id") if isinstance(row.get("meta"), dict) else None
    pool    = fewshot_pool[fewshot_pool["domain"] == domain]
    if cur_id:
        pool = pool[pool["id"] != cur_id]
    shots   = pool.sample(min(NUM_FEWSHOT, len(pool)), random_state=42)
    prompt  = ""
    for _, s in shots.iterrows():
        prompt += fmt(s, with_ans=True) + "\n\n"
    prompt += fmt(row, with_ans=False)
    return prompt

def load_4bit(path, label):
    print(f"\nЗагружаем [{label}]: {path}")
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    tok   = AutoTokenizer.from_pretrained(path)
    model = AutoModelForCausalLM.from_pretrained(path, quantization_config=bnb, device_map="auto")
    model.eval()
    return model, tok

def generate(model, tok, prompt, max_new=50):
    inp = tok(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inp,
            max_new_tokens=max_new,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
            repetition_penalty=1.2,
        )
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Прогоняем каждую модель по отобранным примерам
model_configs = [
    ("BASE",       BASE_MODEL_ID),
    ("SFT",        SFT_CHECKPOINT),
    ("DPO b=0.01", DPO_CHECKPOINTS[0.01]),
    ("DPO b=0.1",  DPO_CHECKPOINTS[0.1]),
    ("DPO b=0.5",  DPO_CHECKPOINTS[0.5]),
]

generations = {label: {} for label, _ in model_configs}

for model_label, model_path in model_configs:
    m, tok = load_4bit(model_path, model_label)
    for qid in sample_ids:
        if qid not in id_to_row.index:
            continue
        row = id_to_row.loc[qid]
        prompt = build_eval_prompt_manual(row, pub)
        generations[model_label][qid] = generate(m, tok, prompt)
    del m, tok
    gc.collect()
    torch.cuda.empty_cache()


pred_col_map = {
    "BASE":       "base_pred",
    "SFT":        "sft_pred",
    "DPO b=0.01": "dpo001_pred",
    "DPO b=0.1":  "dpo01_pred",
    "DPO b=0.5":  "dpo05_pred",
}

print("\n" + "=" * 70)
print("РУЧНОЙ АНАЛИЗ")
print("=" * 70)

for qid in sample_ids:
    if qid not in id_to_row.index:
        continue
    row  = id_to_row.loc[qid]
    mrow = merged[merged["id"] == qid].iloc[0]

    print(f"\nДомен:   {mrow['domain']}")
    print(f"Предмет: {mrow['subject']}")
    print(f"Вопрос:  {row.get('text', '')}")
    print(f"  A) {row.get('option_a', '')}")
    print(f"  B) {row.get('option_b', '')}")
    print(f"  C) {row.get('option_c', '')}")
    print(f"  D) {row.get('option_d', '')}")
    print(f"Правильный ответ: {mrow['gold']}")
    print()

    for model_label, _ in model_configs:
        pred    = mrow[pred_col_map[model_label]]
        ok      = "верно" if pred == mrow["gold"] else "ошибка"
        gentext = generations[model_label].get(qid, "—")
        print(f"  [{model_label:<12}]  логит: {pred} ({ok})  |  генерация: {repr(gentext[:150])}")

    print("-" * 70)

print("\nАнализ завершён.")

BASE:        166 строк
SFT:         166 строк
DPO b=0.01: 166 строк
DPO b=0.1: 166 строк
DPO b=0.5: 166 строк
Примеров с хотя бы одной ошибкой: 146
Отобрано для показа: 10

Загружаем ruMMLU для текста вопросов...
Датасет загружен: 10033 примеров

Загружаем [BASE]: google/gemma-3-4b-it


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Загружаем [SFT]: /content/drive/MyDrive/Colab Notebooks/gemma3_sft/checkpoints/epoch06_hard_ep2of2


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


Загружаем [DPO b=0.5]: /content/drive/MyDrive/Colab Notebooks/gemma3_dpo/beta_0_5


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


РУЧНОЙ АНАЛИЗ

Домен:   college_medicine
Предмет: nan
Вопрос:  Ожидаемым побочным эффектом добавок креатина является:
  A) мышечная слабость.
  B) увеличение массы тела.
  C) мышечные спазмы.
  D) потеря электролитов.
Правильный ответ: B

  [BASE        ]  логит: A (ошибка)  |  генерация: 'D\n\nНайти правильный ответ на вопрос по теме медицины в задании ниже. В качестве ответа запишите только одну букву выбранного варианта: A, B, C или D б'
  [SFT         ]  логит: A (ошибка)  |  генерация: 'A'
  [DPO b=0.01  ]  логит: C (ошибка)  |  генерация: 'A'
  [DPO b=0.1   ]  логит: A (ошибка)  |  генерация: 'A'
  [DPO b=0.5   ]  логит: A (ошибка)  |  генерация: 'A'
----------------------------------------------------------------------

Домен:   college_biology
Предмет: nan
Вопрос:  У садового гороха аллель для высоких растений (D) полностью доминирует над аллелем для карликовых растений (d), а аллель для фиолетовой окраски цветков (W) полностью доминирует над аллелем для белой окраски цветков 

Видно, что ответы через логиты и через генерацию очень часто отличаются!


**Про логиты подробнее**

Когда модель обрабатывает промпт, на выходе последнего слоя она выдаёт вектор размером с весь словарь, это и есть логиты: сырые "оценки уверенности" для каждого токена словаря. Чем больше число — тем выше вероятность что именно этот токен должен идти следующим.

Для оценки на MMLU мы не генерируем текст, а просто смотрим на логиты четырёх конкретных токенов: A, B, C, D. Берём `argmax` — какая буква получила наибольший логит, та и считается ответом.

Генерация зависит от кучи факторов — `repetition_penalty`, длины контекста, того какие буквы уже встречались в промпте, в то время как логиты не зависят.

В нашем случае при оценке логитами модели различались (BASE 20.5%, SFT 27.1%), а при прямом промпте без few-shot все скатились в A, потому что логиты для A оказались чуть выше для всех моделей при таком промпте, и без контекста few-shot примеров модели не могут калибровать свои ответы.

In [ ]:
# Оценка через генерацию
def extract_answer(text: str):
    cyrillic_map = {"А": "A", "В": "B", "С": "C", "Д": "D",
                    "а": "A", "в": "B", "с": "C", "д": "D"}
    text = text.strip()
    if not text:
        return None
    first = text[0]
    first = cyrillic_map.get(first, first).upper()
    if first in ANSWER_LETTERS:
        return first
    for char in text:
        c = cyrillic_map.get(char, char).upper()
        if c in ANSWER_LETTERS:
            return c
    return None


def evaluate_with_generation(model, tokenizer, eval_df, fewshot_pool, label=""):
    results = []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=f"Gen [{label}]"):
        prompt = build_eval_prompt(row, fewshot_pool)
        inputs = tokenizer(
            prompt, return_tensors="pt", truncation=True, max_length=2048
        ).to(model.device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=4,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.2,
            )

        generated = tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        ).strip()

        predicted = extract_answer(generated)
        gold      = str(row.get("outputs", "")).strip().upper()
        meta_id   = row["meta"].get("id") if isinstance(row.get("meta"), dict) else None

        results.append({
            "id":        meta_id,
            "domain":    row.get("domain", ""),
            "subject":   row.get("subject", ""),
            "gold":      gold,
            "generated": generated,
            "predicted": predicted,
            "correct":   predicted == gold if predicted else False,
            "no_answer": predicted is None,
        })

    return pd.DataFrame(results)


def print_results_gen(df, label=""):
    total    = len(df)
    answered = df[~df["no_answer"]]
    no_ans   = df["no_answer"].sum()
    acc_all  = df["correct"].mean()
    acc_ans  = answered["correct"].mean() if len(answered) > 0 else 0.0

    print(f"\n{'='*55}")
    print(f"Результаты (генерация): {label}")
    print(f"{'='*55}")
    print(f"Accuracy (все):         {acc_all*100:.1f}%  |  n={total}")
    print(f"Accuracy (с ответом):   {acc_ans*100:.1f}%  |  n={len(answered)}")
    print(f"Без ответа:             {no_ans} ({no_ans/total*100:.1f}%)")
    print(f"Распределение:          {df['predicted'].value_counts(dropna=False).to_dict()}")
    by_domain = (
        df.groupby("domain")["correct"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "accuracy", "count": "n"})
        .sort_values("accuracy", ascending=False)
    )
    print(f"\nПо доменам:")
    print(by_domain.to_string(float_format=lambda x: f"{x:.3f}"))
    return acc_all


# Прогон всех 5 моделей
model_configs = [
    ("BASE",       BASE_MODEL_ID),
    ("SFT",        SFT_CHECKPOINT),
    ("DPO b=0.01", f"{DPO_OUTPUT_DIR}/beta_0_01"),
    ("DPO b=0.1",  f"{DPO_OUTPUT_DIR}/beta_0_1"),
    ("DPO b=0.5",  f"{DPO_OUTPUT_DIR}/beta_0_5"),
]

gen_results = {}

for model_label, model_path in model_configs:
    m, tok = load_model_for_eval(model_path, label=model_label)

    all_df = evaluate_with_generation(m, tok, eval_df,  clean_eval_df, label=f"{model_label}_all")
    sci_df = evaluate_with_generation(m, tok, eval_sci, clean_eval_df, label=f"{model_label}_sci")

    acc_all = print_results_gen(all_df, f"{model_label} все предметы")
    acc_sci = print_results_gen(sci_df, f"{model_label} естественные науки")

    fname = model_label.replace(" ", "_").replace("=", "").replace(".", "")
    all_df.to_csv(f"{RESULTS_DIR}/gen_{fname}_all.csv", index=False)
    sci_df.to_csv(f"{RESULTS_DIR}/gen_{fname}_sci.csv", index=False)

    gen_results[model_label] = {"acc_all": acc_all, "acc_sci": acc_sci}

    del m, tok
    gc.collect()
    torch.cuda.empty_cache()

#Итоговая таблица
print("\n" + "="*65)
print("ИТОГ (генерация)")
print("="*65)
print(f"{'Модель':<25} {'Все предметы':>15} {'Естественные науки':>20}")
print("-"*65)
for label, res in gen_results.items():
    print(f"{label:<25} {res['acc_all']*100:>14.1f}% {res['acc_sci']*100:>19.1f}%")
print("="*65)


Загружаем [BASE]: google/gemma-3-4b-it


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Gen [BASE_all]:   0%|          | 0/1000 [00:00<?, ?it/s]

Gen [BASE_sci]:   0%|          | 0/166 [00:00<?, ?it/s]


Результаты (генерация): BASE все предметы
Accuracy (все):         23.1%  |  n=1000
Accuracy (с ответом):   23.1%  |  n=1000
Без ответа:             0 (0.0%)
Распределение:          {'A': 391, 'B': 280, 'C': 278, 'D': 51}

По доменам:
                                     accuracy   n
domain                                           
college_computer_science                1.000   1
high_school_european_history            0.500   8
college_physics                         0.500   4
human_sexuality                         0.500   6
world_religions                         0.462  13
philosophy                              0.429  21
high_school_us_history                  0.417  12
abstract_algebra                        0.400   5
global_facts                            0.400  10
machine_learning                        0.400   5
conceptual_physics                      0.385  13
high_school_mathematics                 0.375  24
econometrics                            0.375   8
professional_ac

...

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Gen [DPO b=0.5_all]:   0%|          | 0/1000 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Gen [DPO b=0.5_sci]:   0%|          | 0/166 [00:00<?, ?it/s]


Результаты (генерация): DPO b=0.5 все предметы
Accuracy (все):         24.7%  |  n=1000
Accuracy (с ответом):   24.7%  |  n=1000
Без ответа:             0 (0.0%)
Распределение:          {'C': 670, 'A': 146, 'B': 137, 'D': 47}

По доменам:
                                     accuracy   n
domain                                           
college_computer_science                1.000   1
high_school_european_history            0.750   8
college_physics                         0.500   4
computer_security                       0.500   4
conceptual_physics                      0.462  13
high_school_biology                     0.435  23
astronomy                               0.429  14
international_law                       0.417  12
philosophy                              0.381  21
econometrics                            0.375   8
business_ethics                         0.375   8
professional_accounting                 0.353  17
college_medicine                        0.333   3
human_sexu

### **Результаты**

| Модель | Логиты (все) | Логиты (ест. науки) | Генерация (все) | Генерация (ест. науки) |
|---|:---:|:---:|:---:|:---:|
| **Base** | 23.3% | 20.5% | 23.1% | 21.1% |
| **SFT** | 24.4% | 27.1% | 24.4% | 27.7% |
| **DPO β=0.01** | 24.2% | 26.5% | 0.0% | 0.0% |
| **DPO β=0.1** | 23.7% | 27.1% | 24.2% | 27.7% |
| **DPO β=0.5** | 24.0% | 26.5% | 24.7% | 28.9% |

DPO b=0.01 полностью сломан в генерации — 0% и 1000 пустых ответов. При этом по логитам он давал 24.2%, это означает что модель знает правильный токен (логиты это видят), но генерировать текст не может, — видимо, при beta=0.01 модель слишком сильно отошла от базы и разучилась генерировать связный текст, это подтверждается тем, что при ручном анализе у неё были пустые строки в генерации. При очень низкой beta DPO почти не штрафует за отклонение от reference-модели (SFT), то есть оптимизация почти не ограничена. Модель агрессивно подгоняется под chosen/rejected сигнал и перекалибрует распределение вероятностей так, что логиты для букв A/B/C/D остаются высокими в нужных позициях, но генерация деградирует, модель выучивает выдавать конец последовательности сразу после промпта, потому что в DPO-данных chosen-ответы это одиночные буквы, а rejected тоже одиночные буквы. При низкой beta модель находит самый простой способ максимизировать разницу между chosen и rejected — вообще ничего не генерировать, тогда повторение rejected-буквы точно не произойдёт. Логиты при этом не затронуты, потому что они измеряются до генерации.

DPO b=0.5 лучший по генерации на естественных науках (28.9%), и это при том что по логитам он был слабее SFT. Менее агрессивное DPO-обучение (высокая beta = меньше отклонение от SFT) улучшает генерацию, но не логиты.

SFT и DPO b=0.1 одинаковы по обоим методам, DPO b=0.1 не дал ничего поверх SFT.

BASE на естественных науках хуже всех что ожидаемо, дообучение на данных по доменам естественным наук помогло всем остальным моделям.

Лучшая модель для science-задач по генерации — DPO b=0.5 (28.9%), но отрыв от SFT (27.7%) небольшой — 1.2%, что на 166 примерах это примерно 2 вопроса, статистически это на грани значимости. Реальный вывод такой, что SFT дал основной прирост, DPO на таком маленьком датасете даёт маргинальное улучшение в лучшем случае и ломает модель в худшем (b=0.01). Если продолжать эксперименты, имеет смысл либо увеличить DPO-датасет, либо попробовать более широкий диапазон beta между 0.1 и 0.5.



### **Limitations**

1. Размер DPO-датасета. Основной эксперимент обучался на 500 примерах (dpo_natural_science_500_hard.json), кажется, что это очень мало для DPO. Контрастное обучение требует большего разнообразия пар chosen/rejected, иначе модель может коллапсировать, как при β=0.01. Однако мы провели эксперимент и выяснили, что выборка 2250 примеров тоже не дает желаемого результата. Таким образом, оптимальное соотношение размера выборки и параметра b осталось пока неопределенным.

2. Коллапс генерации при β=0.01. При низкой beta модель нашла короткий и простой путь — генерировать только EOS. По логитам она выглядит адекватно (24.2%), но реальная генерация полностью сломана (0%, 1000 пустых ответов). Это делает β=0.01 вариант непригодным к использованию.

3. Небольшой evaluation сэмпл. Оценка на 166 примерах из естественных наук делает разницу в 1–2% статистически незначимой — разрыв между DPO β=0.5 (28.9%) и SFT (27.7%) составляет буквально примерно 2 вопроса.

4. Узкая область обучения. DPO-датасет покрывает только естественные науки, поэтому модель не улучшается на остальных доменах ruMMLU и потенциально регрессирует на них. Так, например SFT-обучение на данных по доменам естественным наук ухудшило математику, BASE давал 37.5%, SFT только 16.7% на high_school_mathematics.



### **Future improvements/plans**
1. Текущие эксперименты: 0.01, 0.1, 0.5, но имеет смысл протестировать промежуточные значения 0.05, 0.2, 0.3 — особенно между 0.1 и 0.5, где судя по результатам может находится оптимальное значение.

2. Улучшить качество chosen/rejected пар, т.к. контрастность пар напрямую влияет на эффективность DPO.

3. Оценивать на полном evaluation сете. Использовать все доступные примеры ruMMLU вместо сэмпла 1000, чтобы различия в 1–2% были статистически значимы.

4. Провести полный grid search beta на датасете 2250. Сейчас на 2250 примерах протестирован только β=0.1. Учитывая, что лучшим при финальной оценке оказался β=0.5, логичный следующий шаг — прогнать β=0.1, 0.2, 0.3, 0.5 на большом датасете. Вполне возможно, что именно комбинация «большой датасет + высокая beta» даст значимый прирост поверх SFT, которого не удалось достичь в текущих экспериментах.
